In [147]:
import pandas as pd
obesidad = pd.read_csv("Obesidad.csv")
print(obesidad.columns)

Index(['YearStart', 'YearEnd', 'LocationAbbr', 'LocationDesc', 'Datasource',
       'Class', 'Topic', 'Question', 'Data_Value_Unit', 'Data_Value_Type',
       'Data_Value', 'Data_Value_Alt', 'Data_Value_Footnote_Symbol',
       'Data_Value_Footnote', 'Low_Confidence_Limit', 'High_Confidence_Limit ',
       'Sample_Size', 'Total', 'Age(years)', 'Education', 'Gender', 'Income',
       'Race/Ethnicity', 'GeoLocation', 'ClassID', 'TopicID', 'QuestionID',
       'DataValueTypeID', 'LocationID', 'StratificationCategory1',
       'Stratification1', 'StratificationCategoryId1', 'StratificationID1'],
      dtype='object')


In [ ]:
obesidad_recortado = obesidad[['YearStart','LocationAbbr','Topic','Question','Data_Value','Stratification1','StratificationCategory1']]

In [149]:
obesidad_recortado['Topic'].unique()

array(['Obesity / Weight Status', 'Fruits and Vegetables - Behavior',
       'Physical Activity - Behavior'], dtype=object)

In [150]:
# Definimos el mapa de nombres cortos
question_map = {
    'Percent of adults aged 18 years and older who have obesity': 'obesity_pct',
    'Percent of adults aged 18 years and older who have an overweight classification': 'overweight_pct',
    'Percent of adults who report consuming fruit less than one time daily': 'low_fruit_cons_pct',
    'Percent of adults who report consuming vegetables less than one time daily': 'low_veg_cons_pct',
    'Percent of adults who engage in no leisure-time physical activity': 'no_exercise_pct',
    'Percent of adults who achieve at least 150 minutes a week of moderate-intensity aerobic physical activity': 'aerobic_150min_pct',
    'Percent of adults who achieve at least 150 minutes a week of moderate-intensity aerobic physical activity or 75 minutes a week of vigorous-intensity aerobic activity (or an equivalent combination)': 'aerobic_75_150min_pct',
    'Percent of adults who achieve at least 150 minutes a week of moderate-intensity aerobic physical activity or 75 minutes a week of vigorous-intensity aerobic physical activity and engage in muscle-strengthening activities on 2 or more days a week': 'full_guideline_met_pct',
    'Percent of adults who achieve at least 300 minutes a week of moderate-intensity aerobic physical activity or 150 minutes a week of vigorous-intensity aerobic activity (or an equivalent combination)': 'aerobic_300min_pct',
    'Percent of adults who engage in muscle-strengthening activities on 2 or more days a week': 'muscle_strength_pct'
}
obesidad_recortado['Question'] = obesidad_recortado['Question'].replace(question_map)
obesidad_recortado = obesidad_recortado.dropna(subset=['Data_Value'])

/tmp/ipykernel_12076/3859991700.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  obesidad_recortado['Question'] = obesidad_recortado['Question'].replace(question_map)


In [151]:
obesidad_recortado.head(20)

,YearStart,LocationAbbr,Topic,Question,Data_Value,Stratification1,StratificationCategory1
0,2011,AL,Obesity / Weight Status,obesity_pct,32.0,Total,Total
1,2011,AL,Obesity / Weight Status,obesity_pct,32.3,Male,Gender
2,2011,AL,Obesity / Weight Status,obesity_pct,31.8,Female,Gender
3,2011,AL,Obesity / Weight Status,obesity_pct,33.6,Less than high school,Education
4,2011,AL,Obesity / Weight Status,obesity_pct,32.8,High school graduate,Education
5,2011,AL,Obesity / Weight Status,obesity_pct,33.8,Some college or technical school,Education
6,2011,AL,Obesity / Weight Status,obesity_pct,26.4,College graduate,Education
7,2011,AL,Obesity / Weight Status,obesity_pct,16.3,18 - 24,Age (years)
8,2011,AL,Obesity / Weight Status,obesity_pct,35.2,25 - 34,Age (years)
9,2011,AL,Obesity / Weight Status,obesity_pct,35.5,35 - 44,Age (years)


In [152]:
df_pivot = obesidad_recortado.pivot_table(
    index=['YearStart', 'LocationAbbr', 'Stratification1'],
    columns='Question',
    values='Data_Value'
).reset_index()

df_pivot.columns.name = None  
    

In [153]:
df_pivot.dropna(inplace=True)
df_pivot.drop(columns=['LocationAbbr'], inplace=True)
df_pivot = pd.get_dummies(df_pivot, drop_first=True, dtype=int)
df_pivot.info()


<class 'pandas.core.frame.DataFrame'>
Index: 3980 entries, 0 to 6753
Data columns (total 37 columns):
 #   Column                                            Non-Null Count  Dtype  
---  ------                                            --------------  -----  
 0   YearStart                                         3980 non-null   int64  
 1   aerobic_300min_pct                                3980 non-null   float64
 2   aerobic_75_150min_pct                             3980 non-null   float64
 3   full_guideline_met_pct                            3980 non-null   float64
 4   low_fruit_cons_pct                                3980 non-null   float64
 5   low_veg_cons_pct                                  3980 non-null   float64
 6   muscle_strength_pct                               3980 non-null   float64
 7   no_exercise_pct                                   3980 non-null   float64
 8   obesity_pct                                       3980 non-null   float64
 9   overweight_pct          

In [154]:
df_pivot.corr(numeric_only=True)['obesity_pct'].sort_values(ascending=False)[1:]

no_exercise_pct                                     0.521588
low_fruit_cons_pct                                  0.406824
low_veg_cons_pct                                    0.362880
Stratification1_Non-Hispanic Black                  0.214919
Stratification1_American Indian/Alaska Native       0.158699
Stratification1_45 - 54                             0.147243
Stratification1_Less than $15,000                   0.143254
Stratification1_55 - 64                             0.136903
Stratification1_Less than high school               0.119526
YearStart                                           0.105250
Stratification1_35 - 44                             0.098351
Stratification1_Hawaiian/Pacific Islander           0.062102
Stratification1_High school graduate                0.061275
Stratification1_$25,000 - $34,999                   0.054658
Stratification1_2 or more races                     0.050351
overweight_pct                                      0.042154
Stratification1_Hispanic

In [155]:
from sklearn.datasets import make_moons
from sklearn.tree import DecisionTreeRegressor
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

from sklearn.tree import export_graphviz
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, mean_absolute_error,mean_squared_error

In [156]:
X = df_pivot.drop(['obesity_pct'],axis=1)
y = df_pivot['obesity_pct'].to_frame()

In [157]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

In [158]:
tree_clf = DecisionTreeRegressor()
tree_clf.fit(X_train, y_train)

# For a regressor there is no `classes_` attribute, so omit `class_names`.
export_graphviz(
    tree_clf,
    out_file="./obesity.dot",
    feature_names=list(df_pivot.drop(['obesity_pct'], axis=1).columns),
    rounded=True,
    filled=True
)

# If "dot: command not found" occurs, install graphviz in the system:
# sudo apt install graphviz
!dot -Tpng obesity.dot -o obesity.png

dot: graph is too large for cairo-renderer bitmaps. Scaling by 0.160801 to fit


In [159]:
y_pred = tree_clf.predict(X_test)
#print(y_pred)
print("Mean Absolute Error:", mean_absolute_error(y_test, y_pred))
print("R^2 Score:", tree_clf.score(X_test, y_test))
print("Mean Squared Error:", mean_squared_error(y_test,y_pred))

Mean Absolute Error: 3.688140703517588
R^2 Score: 0.41608424681767453
Mean Squared Error: 26.591065326633167


In [160]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [161]:
svm = SVR(kernel="rbf", coef0=0.1, C=1)

svm.fit(X_train_scaled, y_train)

pred_rbf = svm.predict(X_test_scaled)

print("Mean Absolute Error:", mean_absolute_error(y_test, pred_rbf))
print("R^2 Score:", svm.score(X_test_scaled, y_test))
print("Mean Squared Error:", mean_squared_error(y_test,pred_rbf))

/home/ciabd12/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Mean Absolute Error: 2.4904683073330163
R^2 Score: 0.7355560955172215
Mean Squared Error: 12.042567957805787


In [162]:
rf = RandomForestRegressor(n_estimators=700, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)


/home/ciabd12/anaconda3/lib/python3.13/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


RandomForestRegressor(n_estimators=700, n_jobs=-1, random_state=42)

In [163]:
pred_rf = rf.predict(X_test_scaled)

print("Mean Absolute Error:", mean_absolute_error(y_test, pred_rf))
print("R^2 Score:", rf.score(X_test_scaled, y_test))
print("Mean Squared Error:", mean_squared_error(y_test,pred_rf))

Mean Absolute Error: 4.6038882986360425
R^2 Score: 0.2662743790682963
Mean Squared Error: 33.41328918030976


/home/ciabd12/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/home/ciabd12/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
